In [1]:
import os
os.environ['USER_AGENT'] = 'BA_GPT/1.0 (contact: walrus_z@hotmail.com)'

In [2]:
import os
import sys
import yaml
import types
import pprint
import logging
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.memory.canvas import TextCanvasMemory
from autogen_ext.models.openai import OpenAIChatCompletionClient
from opentelemetry import trace, _events
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk._events import EventLoggerProvider
from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
from opentelemetry.instrumentation.openai import OpenAIInstrumentor


# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"
os.environ["AZURE_SDK_TRACING_IMPLEMENTATION"] = "opentelemetry"

from agents_creator import create_agents_from_config
from rag_helpers import rag_indexer_runner
from model_loader import load_model_client
from agents_tools_helpers import selector_func

In [3]:
logging.basicConfig(level=logging.WARNING)

# For trace logging.
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.addHandler(logging.StreamHandler())
trace_logger.setLevel(logging.DEBUG)

# For structured message logging, such as low-level messages between agents.
event_logger = logging.getLogger(EVENT_LOGGER_NAME)
event_logger.addHandler(logging.StreamHandler())
event_logger.setLevel(logging.DEBUG)

In [4]:
# Set up resource
resource = Resource(attributes={
    "service.name": "autogen-test-agentchat"
})

# Set up tracer provider
trace.set_tracer_provider(TracerProvider(resource=resource))

# Configure OTLP exporter
otlp_exporter = OTLPSpanExporter(
    endpoint="http://localhost:4318/v1/traces"
)

# Add span processor
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(otlp_exporter)
)

# Set up logger provider
logger_provider = LoggerProvider(resource=resource)
logger_provider.add_log_record_processor(
    BatchLogRecordProcessor(OTLPLogExporter(endpoint="http://localhost:4318/v1/logs"))
)
_events.set_event_logger_provider(EventLoggerProvider(logger_provider))

# Enable OpenAI instrumentation
OpenAIInstrumentor().instrument()

ERROR:opentelemetry.sdk._shared_internal:Exception while exporting Span.
Traceback (most recent call last):
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py", line 787, 

In [5]:
rag_vector_memory = await rag_indexer_runner(lib_path + '/rag_config.yaml')

✅ All documents and URLs are already indexed.


In [6]:
# 1. Create the canvas memory and tool:
text_canvas_memory = TextCanvasMemory()
update_file_tool = text_canvas_memory.get_update_file_tool()

# 2. Register the tool as 'canvas.update_file_tool' for YAML import:
canvas = types.SimpleNamespace()
canvas.update_file_tool = update_file_tool
sys.modules['canvas'] = canvas  # This enables the tool import by your agent factory!

In [7]:
model_client = load_model_client("azure")

In [8]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY', 'azure_endpoint': 'https://224d2-m9kjdmy0-eastus2.openai.azure.com/', 'azure_deployment': 'gpt-4o', 'api_version': '2024-12-01-preview'}
create_args: {'model': 'gpt-4o'}


In [9]:
# Construct the absolute path to project/lib/agents_config.yaml user_proxy agent
AGENTS_CONFIG = os.path.abspath(os.path.join(notebook_dir, '..', 'lib', 'agents_config.yaml'))
with open(AGENTS_CONFIG, "r") as f:
        config_yaml = yaml.safe_load(f)
agent_names = list(config_yaml.keys())

user, agents = await create_agents_from_config(
    AGENTS_CONFIG,
    agent_names=agent_names,
    model_client=model_client,
    vector_memory=rag_vector_memory,
    canvas_memory=text_canvas_memory,
    input_func=input
)

In [10]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    pprint.pprint(vars(agent))




--- Agent: planning_agent ---
{'_description': 'The orchestrator agent responsible for breaking down a new '
                 'task into subtasks, assigning them to the right agents, and '
                 'ensuring the workflow progresses smoothly until completion.',
 '_handoff_tools': [],
 '_handoffs': {},
 '_is_running': False,
 '_max_tool_iterations': 1,
 '_memory': None,
 '_metadata': {},
 '_model_client': <autogen_ext.models.openai._openai_client.AzureOpenAIChatCompletionClient object at 0x7f404ce43770>,
 '_model_client_stream': True,
 '_model_context': <autogen_core.model_context._unbounded_chat_completion_context.UnboundedChatCompletionContext object at 0x7f404cdb02f0>,
 '_name': 'planning_agent',
 '_output_content_type': None,
 '_output_content_type_format': None,
 '_reflect_on_tool_use': False,
 '_structured_message_factory': None,
 '_system_messages': [SystemMessage(content='You are the PLANNING AGENT, the coordinator for this multi-agent system.\n\n**YOUR MISSION**\n  Brea

In [11]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    print("Tools:")
    for tool in agent._tools:
        print(f"Tool Name: {tool.name}")
        print(f"Description: {tool.description}\n")



--- Agent: planning_agent ---
Tools:

--- Agent: interviewer_agent ---
Tools:
Tool Name: save_conversation_to_jsonl
Description: 
Saves either:
1) A Q&A pair (interviewer + user) in one atomic JSONL write, OR
2) A generic list of records.



--- Agent: summarizer_agent ---
Tools:

--- Agent: functional_spec_writer_agent ---
Tools:
Tool Name: write_md_file_tool
Description: 
Writes the functional specification to a Markdown file.


Tool Name: use_case_agent
Description: An agent dedicated to generating Use Cases and User Stories from the project information.

Tool Name: glossary_agent
Description: An agent dedicated to building a Glossary of terms from the gathered requirements and project information.


--- Agent: functional_spec_checker_agent ---
Tools:
Tool Name: read_md_file_tool
Description: 
Reads the entire content of a Markdown file and returns it as a string.

Args:
    file_path (str): Optional path to the Markdown spec (defaults to DEFAULT_MD_SPEC_PATH).

Returns:
    str: T

In [12]:
selector_prompt = """
Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [13]:
# Order agents (workflow order), 
agent_order = [
    "planning_agent",
    "interviewer_agent",
    "summarizer_agent",
    "functional_spec_writer_agent",
    "functional_spec_checker_agent",
    "diagram_creator_agent",
    "mermaid_code_reviewer_agent",
    "terminate_agent"
]
member_agents = [agents[name] for name in agent_order if name in agents] + [user]

In [14]:
termination = MaxMessageTermination(100000000) | TextMentionTermination("TERMINATE")

# --- Use SelectorGroupChat ---
team = SelectorGroupChat(
    member_agents,
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
    model_client_streaming=False,
    selector_func=selector_func,
)

In [15]:
task = "Interview a customer to collect requirements and scope of a new product then write the functional specifications"

In [16]:
runtime = SingleThreadedAgentRuntime(
            tracer_provider=trace.NoOpTracerProvider(),  # Disable telemetry for runtime.
        )
runtime.start()

In [17]:
try:
    await Console(team.run_stream(task=task))
except Exception as e:
    print(f"Error during team run_stream: {e}")

await runtime.stop()

try:
    await model_client.close()
except Exception as e:
    print(f"Error closing model_client: {e}")

---------- TextMessage (user) ----------
Interview a customer to collect requirements and scope of a new product then write the functional specifications
---------- ModelClientStreamingChunkEvent (planning_agent) ----------
Task breakdown initiated:

1. Begin by gathering requirements and product scope from the customer. This responsibility falls to the interviewer_agent.
2. Summarize the collected requirements for clarity and better understanding (summarizer_agent).
3. Draft the functional specification document (functional_spec_writer_agent).
4. Review the functional specification for completeness (functional_spec_checker_agent).
5. Create process flow diagrams if required (diagram_creator_agent).
6. Review any Mermaid code diagrams for correctness (mermaid_code_reviewer_agent).
7. Conclude the task and terminate the process (terminate_agent).

Task Status:
interviewer_agent: not started  
summarizer_agent: not started  
functional_spec_writer_agent: not started  
functional_spec_che

id='e51f4f50-8d1e-46f7-a00e-f2ff973c5ae8' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 38, 23, 788990, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_LnVpZKNolBTpZ3NIMcKGfmJd', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Could you provide a high-level overview of the new product, including its purpose and intended goal?","subject":"Product Overview","q_index":1},{"speaker":"user","message":"The scope of the project is to design and develop an online ecommerce website for apparels. Customers will be able to use the frontend website for searching the apparels and place the order by making online payment of the order in advance. Web based admin panel/ backend will help admin user i.e. business owner to manage the products, categories, prices and orders placed by the customers. Objectives: Enhancing client’s offline business of apparels b

---------- ToolCallRequestEvent (interviewer_agent) ----------
[FunctionCall(id='call_LnVpZKNolBTpZ3NIMcKGfmJd', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Could you provide a high-level overview of the new product, including its purpose and intended goal?","subject":"Product Overview","q_index":1},{"speaker":"user","message":"The scope of the project is to design and develop an online ecommerce website for apparels. Customers will be able to use the frontend website for searching the apparels and place the order by making online payment of the order in advance. Web based admin panel/ backend will help admin user i.e. business owner to manage the products, categories, prices and orders placed by the customers. Objectives: Enhancing client’s offline business of apparels by transforming it into online ecommerce website., Making online ecommerce platform for buyers to buy apparels from website., Customers/ buyers will be able to search for the various 

id='87f76ac1-057a-41ae-97f4-cdce2ceed082' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 38, 23, 794028, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_LnVpZKNolBTpZ3NIMcKGfmJd', is_error=False)] type='ToolCallExecutionEvent'
id='87f76ac1-057a-41ae-97f4-cdce2ceed082' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 38, 23, 794028, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_LnVpZKNolBTpZ3NIMcKGfmJd', is_error=False)] type='ToolCallExecutionEvent'
DEBUG:autogen_agentchat.events:id='87f76ac1-057a-41ae-97f4-cdce2ceed082' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 38

---------- ToolCallExecutionEvent (interviewer_agent) ----------
[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_LnVpZKNolBTpZ3NIMcKGfmJd', is_error=False)]
---------- ModelClientStreamingChunkEvent (interviewer_agent) ----------
- Subject Tracking Status:
    Product Overview  asked=1, status=pending
    KPI & Success Metrics  asked=0, status=not started
    Scope In and Out  asked=0, status=not started
    Non-Functional Requirements  asked=0, status=not started
    User Roles & Personas  asked=0, status=not started
    Integrations & External Systems  asked=0, status=not started
    Constraints & Assumptions  asked=0, status=not started
    Dependencies & Risks  asked=0, status=not started

QUESTION: What key performance indicators (KPIs) or success metrics should we consider to ensure the product meets its goals effectively?
---------- ModelClientStreamingChunkEvent (planning_agent) ----------

id='eb4dcb0d-435a-40b9-ba3e-1421df785411' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 38, 47, 171401, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_qDoKbr4MZMVJ6EMaqfKqlnKm', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: What key performance indicators (KPIs) or success metrics should we consider to ensure the product meets its goals effectively?","subject":"KPI & Success Metrics","q_index":1},{"speaker":"user","message":"Enhancing client’s offline business of apparels by transforming it into online ecommerce website., Making online ecommerce platform for buyers to buy apparels from website., Customers/ buyers will be able to search for the various kinds of apparels for e.g. Shirts. Jeans etc, and will be able to order it from website by making online payment of products. Business owner will be able to manage his products by category 

... (85 outputs left out)

id='a538add7-3363-45da-9c8c-2eadd2e05dd2' source='terminate_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 52, 7, 569718, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_FxNOro5gnVuekC9b7N8AgPHM', arguments='{"session_dir":"project_root/experience/sess-20231101105530","session_id":"sess-20231101105530"}', name='save_experience_chroma')] type='ToolCallRequestEvent'
id='a538add7-3363-45da-9c8c-2eadd2e05dd2' source='terminate_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 52, 7, 569718, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_FxNOro5gnVuekC9b7N8AgPHM', arguments='{"session_dir":"project_root/experience/sess-20231101105530","session_id":"sess-20231101105530"}', name='save_experience_chroma')] type='ToolCallRequestEvent'
DEBUG:autogen_agentchat.events:id='a538add7-3363-45da-9c8c-2eadd2e05dd2' sou

---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_FxNOro5gnVuekC9b7N8AgPHM', arguments='{"session_dir":"project_root/experience/sess-20231101105530","session_id":"sess-20231101105530"}', name='save_experience_chroma')]


id='377ff2bd-5439-4f72-a490-a68f046da7e1' source='terminate_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 52, 14, 843356, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"ok\\":true,\\"session_id\\":\\"sess-20231101105530\\",\\"files_indexed\\":0,\\"written_items\\":2,\\"collection\\":\\"tfm_ba_gpt_experiences\\",\\"persistence_path\\":\\"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.experiences\\"}", "annotations": null}]', name='save_experience_chroma', call_id='call_FxNOro5gnVuekC9b7N8AgPHM', is_error=False)] type='ToolCallExecutionEvent'
id='377ff2bd-5439-4f72-a490-a68f046da7e1' source='terminate_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 14, 15, 52, 14, 843356, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"ok\\":true,\\"session_id\\":\\"sess-20231101105530\\",\\"files_indexed\\":0,\\"written_items\\":2,\\"colle

---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"ok\\":true,\\"session_id\\":\\"sess-20231101105530\\",\\"files_indexed\\":0,\\"written_items\\":2,\\"collection\\":\\"tfm_ba_gpt_experiences\\",\\"persistence_path\\":\\"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.experiences\\"}", "annotations": null}]', name='save_experience_chroma', call_id='call_FxNOro5gnVuekC9b7N8AgPHM', is_error=False)]
---------- ModelClientStreamingChunkEvent (terminate_agent) ----------
**STEP 8**: Experience metadata successfully saved for session `sess-20231101105530`.  

All steps completed successfully.  

**TERMINATE**
